<a href="https://colab.research.google.com/github/ndukwegodslove-del/FlyrankAI-MLinternship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install huggingface_hub

In [2]:
from google.colab import userdata
from huggingface_hub import whoami

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

print("Successfully authenticated!")
print("Hugging Face account:", whoami(token=HF_TOKEN)["name"])

Successfully authenticated!
Hugging Face account: Godslove888


In [3]:
%pip -q install duckdb huggingface_hub

In [4]:
import duckdb

# Create an in-memory DuckDB connection
con = duckdb.connect()

# Give DuckDB access to your Hugging Face token
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

# Define the warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

# Define our warehouse tables
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("DuckDB connected successfully!")

DuckDB connected successfully!


In [5]:
# Test access to the clients table
client_count = con.sql(
    f"SELECT COUNT(*) FROM {TABLES['dim_clients']}"
).fetchone()[0]

print("Number of clients:", client_count)

Number of clients: 104


In [6]:
schema = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}"
).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Unit of analysis

The original unit of analysis in `fact_content_daily_performance` is one content page belonging to one client, measured on one reporting date.

The unique grain is:

`report_date + client_hash_id + content_hash_id`

For this Refresh / Content Opportunity Scoring exercise, the intended ranking unit is one content page. Daily records will therefore be aggregated to produce page-level measurements for prioritising content review.

### Time window

The selected observation period is 1 March 2026 to 31 March 2026.

March 2026 provides the historical performance window for the initial analysis. The intended ranking decision will be made after this observation period, using only information available by that decision time.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Features

The following fields are potential features because they describe observed performance during the historical observation period:

- `gsc_impressions`: measures how often a page appeared in Google Search results.
- `gsc_clicks`: measures clicks received from Google Search.
- `gsc_avg_position`: measures the average search position.
- `ga4_sessions`: measures website sessions associated with a page.
- `ga4_engaged_sessions`: measures engaged sessions associated with a page.

These measurements will be aggregated over the observation window. Their availability flags must be respected when interpreting missing or zero-filled measurements.

**Label / Target:**

`is_declining_label` is a binary proxy for subsequent search-performance decline. It is derived by comparing each page-client pair's total GSC impressions in March 2026 with its total GSC impressions in April 2026.

A value of `1` indicates that April impressions declined by more than 20% compared with March impressions. A value of `0` indicates that the decline did not exceed 20%.

Only page-client pairs with at least 25 observed GSC reporting days in both months are included in the labelled dataset. April data is used exclusively to construct the outcome label and must not be used as a predictive feature.
### Context

- `client_hash_id`: identifies the client and supports grouping.
- `content_hash_id`: identifies the content page.
- `report_date`: identifies the observation date.
- `month`: identifies the monthly partition.

These fields help with filtering, grouping and joining but will not be used as predictive features.

### Excluded

- `trend_pct` and `trend_direction`: derived performance indicators that must not be used as features when they form part of the outcome definition.
- `client_has_gsc` and `client_has_ga4`: source-level availability information, not page-performance measurements.
- `gsc_data_available` and `ga4_data_available`: used to determine whether measurements are valid, not as substitutes for actual performance.
- Client names, domains, URLs and private search queries: excluded to protect privacy.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS duplicate_count
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

display(grain_check)

print("Duplicate combinations found:", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


Duplicate combinations found: 0


In [8]:
march_summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

display(march_summary)

,total_rows,unique_clients,unique_content_items,earliest_date,latest_date
0,9841378,55,331437,2026-03-01,2026-03-31


The March 2026 partition contains 9,841,378 daily performance records, covering 55 unique clients and 331,437 unique content items. The reporting dates span 1 March to 31 March 2026.

In [9]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows

    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


The availability check shows that Google Search Console data is available for 3,611,061 records, while Google Analytics 4 data is available for 413,966 records. Only 364,347 records have both sources available. This demonstrates that data availability varies across records. Consequently, unavailable measurements must not automatically be interpreted as zero performance.

In [10]:
content_schema = con.sql(f"""
    DESCRIBE SELECT * FROM {TABLES['dim_content']}
""").df()

display(content_schema)

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [13]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,

        SUM(gsc_clicks) AS gsc_clicks,

        SUM(gsc_sum_position) * 1.0 /
            NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position,

        SUM(ga4_sessions) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_sessions,

        SUM(ga4_engaged_sessions) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_engaged_sessions

    FROM {TABLES['fact_daily']}

    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.450877,NaN,NaN
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.298246,NaN,NaN
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,5.637584,4.0,0.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.906404,9.0,0.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,3.950542,3.0,0.0


In [14]:
print("Feature frame shape:", feature_frame.shape)

print("\nMissing values:")
display(feature_frame.isna().sum())

print("\nFeature data types:")
display(feature_frame.dtypes)

Feature frame shape: (176738, 7)

Missing values:


,0
client_hash_id,0
content_hash_id,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_sessions,112882
ga4_engaged_sessions,112882



Feature data types:


,0
client_hash_id,object
content_hash_id,object
gsc_impressions,float64
gsc_clicks,float64
gsc_avg_position,float64
ga4_sessions,float64
ga4_engaged_sessions,float64


### Selected features and availability at decision time

The feature frame contains five performance features aggregated from March 2026 daily records.

1. `gsc_impressions`: Total search impressions during March. Knowable because impressions were recorded during the historical observation period.

2. `gsc_clicks`: Total search clicks during March. Knowable because clicks were recorded during the historical observation period.

3. `gsc_avg_position`: Impression-weighted average search position during March. Knowable because the calculation uses search-position measurements and impressions recorded during the observation period.

4. `ga4_sessions`: Total available GA4 sessions during March. Knowable because the sessions were measured during the historical observation period. Unavailable measurements are retained as missing values.

5. `ga4_engaged_sessions`: Total available engaged sessions during March. Knowable because engagement measurements were recorded during the historical observation period. Unavailable measurements are retained as missing values.

The ranking decision is assumed to occur after March 2026. Therefore, these historical measurements would be available at that decision moment.

### Feature-frame inspection

The resulting feature frame contains 176,738 client-content combinations and seven columns, comprising five performance features and two contextual identifiers.

Both GA4 features have 112,882 missing values. This indicates that GA4 availability is limited across the selected records. Missing values are not interpreted as zero performance.

In [15]:
april_outcomes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions

    FROM {TABLES['fact_daily']}

    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("April page-client records:", len(april_outcomes))
display(april_outcomes.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April page-client records: 194760


,client_hash_id,content_hash_id,april_impressions
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,187.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,23.0
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,17.0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,202.0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,93.0


In [16]:
april_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        COUNT(DISTINCT report_date) AS april_days_observed
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print("April page-client records:", len(april_coverage))
display(april_coverage.head())

print("\\nApril days observed per page:")
display(april_coverage["april_days_observed"].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April page-client records: 194760


,client_hash_id,content_hash_id,april_impressions,april_days_observed
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,187.0,28
1,client_62f4a7e64f5e0096,content_13a8105125458098,23.0,14
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,17.0,12
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,202.0,29
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,93.0,27


\nApril days observed per page:


,april_days_observed
count,194760.000000
mean,20.030088
std,10.985802
min,1.000000
25%,9.000000
50%,25.000000
75%,30.000000
max,30.000000


In [17]:
march_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS march_days_observed
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print("March page-client records:", len(march_coverage))
display(march_coverage["march_days_observed"].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March page-client records: 176738


,march_days_observed
count,176738.000000
mean,20.431718
std,11.480153
min,1.000000
25%,9.000000
50%,26.000000
75%,31.000000
max,31.000000


In [18]:
# Combine March features with March and April coverage/outcomes
labelled_frame = (
    feature_frame
    .merge(
        march_coverage,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
    .merge(
        april_coverage,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)

# Keep only records with sufficient GSC coverage in both months
labelled_frame = labelled_frame[
    (labelled_frame["march_days_observed"] >= 25) &
    (labelled_frame["april_days_observed"] >= 25)
].copy()

# Calculate percentage change in impressions
labelled_frame["impressions_decline_pct"] = (
    (labelled_frame["gsc_impressions"] -
     labelled_frame["april_impressions"])
    / labelled_frame["gsc_impressions"]
) * 100

# Label: 1 means impressions declined by more than 20%
labelled_frame["is_declining_label"] = (
    labelled_frame["impressions_decline_pct"] > 20
).astype(int)

print("Records with sufficient coverage:", len(labelled_frame))
print("\nLabel distribution:")
display(labelled_frame["is_declining_label"].value_counts())

print("\nSample labelled records:")
display(labelled_frame.head())

Records with sufficient coverage: 81742

Label distribution:


,count
is_declining_label,
0,41446
1,40296



Sample labelled records:


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,march_days_observed,april_impressions,april_days_observed,impressions_decline_pct,is_declining_label
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.450877,NaN,NaN,31,1151.0,30,-0.964912,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.298246,NaN,NaN,26,73.0,26,-28.070175,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,5.637584,4.0,0.0,30,98.0,27,34.228188,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.906404,9.0,0.0,31,2275.0,30,-60.098522,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,3.950542,3.0,0.0,31,6266.0,30,-126.209386,0


In [19]:
# Rank pages by March impressions, highest first
baseline_ranking = labelled_frame.sort_values(
    by="gsc_impressions",
    ascending=False
).copy()

# Evaluate the first 50 ranked pages
top_50 = baseline_ranking.head(50)

precision_at_50 = top_50["is_declining_label"].sum() / len(top_50)

print("Baseline ranking: March impressions")
print("Pages evaluated:", len(top_50))
print("Pages that later experienced >20% decline:",
      top_50["is_declining_label"].sum())
print("Precision@50:", round(precision_at_50, 4))
print("Precision@50 (%):", round(precision_at_50 * 100, 2))

Baseline ranking: March impressions
Pages evaluated: 50
Pages that later experienced >20% decline: 17
Precision@50: 0.34
Precision@50 (%): 34.0


In [20]:
# Create a separate copy for the leakage experiment
leaked_ranking = labelled_frame.copy()

# Deliberately introduce the future outcome as a feature
leaked_ranking["leaked_label_feature"] = (
    leaked_ranking["is_declining_label"]
)

# Rank using the leaked information
leaked_ranking = leaked_ranking.sort_values(
    by="leaked_label_feature",
    ascending=False
)

# Evaluate the top 50
leaked_top_50 = leaked_ranking.head(50)

leaked_precision_at_50 = (
    leaked_top_50["is_declining_label"].sum()
    / len(leaked_top_50)
)

print("Leaked ranking experiment")
print("Pages evaluated:", len(leaked_top_50))
print("Pages that experienced >20% decline:",
      leaked_top_50["is_declining_label"].sum())
print("Precision@50:", round(leaked_precision_at_50, 4))
print("Precision@50 (%):", round(leaked_precision_at_50 * 100, 2))

Leaked ranking experiment
Pages evaluated: 50
Pages that experienced >20% decline: 50
Precision@50: 1.0
Precision@50 (%): 100.0


In [21]:
# Remove the deliberately leaked feature
leaked_ranking = leaked_ranking.drop(
    columns=["leaked_label_feature"]
)

print("Leaked feature removed:",
      "leaked_label_feature" not in leaked_ranking.columns)

print("\nRemaining columns:")
print(leaked_ranking.columns.tolist())

Leaked feature removed: True

Remaining columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'march_days_observed', 'april_impressions', 'april_days_observed', 'impressions_decline_pct', 'is_declining_label']


#### Target Leakage Experiment

The honest baseline ranking, which used March impressions, achieved a Precision@50 of 34%. I then deliberately introduced `is_declining_label`, a column derived from April impressions, as a ranking feature. This increased Precision@50 to 100%.

The apparent improvement was caused by target leakage. The ranking process was given information about the future outcome it was supposed to predict. Such information would not have been available at the actual decision time after March 31, 2026.

The leaked feature was subsequently removed. The honest baseline score of 34% remains the valid reference result for this experiment. The 100% result is retained only to demonstrate how leakage can produce misleadingly perfect evaluation results.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limitations

The dataset has several limitations that may affect the reliability and interpretation of the ranking results.

1. **Uneven data availability:** GSC and GA4 measurements are not available for every page or reporting date. Records without available GA4 measurements are treated as missing rather than zero. To improve comparability, the outcome analysis was restricted to page-client pairs with at least 25 observed GSC days in both March and April 2026.

2. **Reduced sample coverage:** Applying the minimum coverage requirement reduced the dataset to 81,742 page-client pairs. This improves consistency but may introduce selection bias because pages with insufficient tracking coverage were excluded.

3. **Missing GA4 measurements:** GA4 sessions and engaged sessions are missing for a substantial proportion of the feature records. Consequently, these features may provide incomplete information for ranking.

4. **Limited observation period:** The analysis uses March 2026 as the feature observation period and April 2026 as the outcome period. A single month of historical observations cannot establish long-term performance patterns or seasonal behaviour.

5. **Proxy outcome:** A decline of more than 20% in observed impressions is used as a proxy for a content opportunity. However, a decline does not necessarily mean that a page needs refreshing or that refreshing it will improve performance.

6. **External influences:** Search algorithm changes, seasonal demand, competition, website changes and measurement inconsistencies may affect impressions. The available data cannot establish the precise cause of an observed decline.

7. **Baseline simplicity:** The 34% Precision@50 result comes from a simple ranking based on March impressions. It is a baseline for comparison, not evidence that a developed ranking model will perform better.

The ranking should therefore be treated as decision support for human review rather than an automatic recommendation or guarantee of improved content performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.